# ERA5 to PRISM Downscaling (Georgia)

Regional climate downscaling pipeline with temporal ERA5 inputs, classical baselines, and a ConvLSTM main model.


## 1. Problem Setup
This project learns a mapping from coarse ERA5 temperature fields to higher-resolution PRISM targets over Georgia.

Formulation: `ERA5(t-k+1 ... t) -> PRISM(t)`.

Modeling direction: a compact regional pipeline aligned with ConvLSTM-style spatiotemporal modeling and motivated by modern weather AI systems (e.g., FourCastNet, GraphCast, Prithvi WxC) without attempting to reproduce foundation-scale models.


## 2. ERA5 vs PRISM
- **ERA5**: coarse-resolution reanalysis input sequence
- **PRISM**: high-resolution observational target

The dataset aligns both sources by date, clips/reprojects PRISM to the ERA5 regional extent, and constructs temporal history windows.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets').exists() and (ROOT.parent / 'datasets').exists():
    ROOT = ROOT.parent

ERA5_PATH = ROOT / 'data_raw/era5_georgia_temp.nc'
PRISM_DIR = ROOT / 'data_raw/prism'
RESULTS_DIR = ROOT / 'results/evaluation'

print(f'Project root: {ROOT}')
print(f'ERA5 available: {ERA5_PATH.exists()}')
print(f'PRISM available: {PRISM_DIR.exists()}')
print(f'Results available: {RESULTS_DIR.exists()}')


## 3. Dataset Overview and Temporal Inputs
Each training sample contains:
- input tensor: `X [T, 1, H_era5, W_era5]`
- target tensor: `Y [1, H_prism, W_prism]`

`T` is controlled by `--history-length` in training/evaluation scripts.


In [ ]:
if ERA5_PATH.exists() and PRISM_DIR.exists():
    try:
        from datasets.prism_dataset import ERA5_PRISM_Dataset
        ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=5, verbose=False)
        x, y = ds[0]
        print(f'Dataset samples: {len(ds)}')
        print(f'X shape: {tuple(x.shape)}')
        print(f'Y shape: {tuple(y.shape)}')
    except Exception as exc:
        print(f'Dataset inspection failed: {exc}')
else:
    print('Data not found. Run:')
    print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
    print('python data_pipeline/download_prism.py --start-date 20230101 --days 3 --variable tmean')


## 4. Model Family
Evaluated model set:
- Persistence baseline
- Global linear baseline
- Spatial CNN baseline
- ConvLSTM temporal downscaler (main model)

Training entrypoint supports `--model cnn` and `--model convlstm`. Evaluation can compare learned and non-learned baselines in one run.


## 5. Results
This section loads saved evaluation artifacts from `results/evaluation/`.


In [ ]:
comparison_files = sorted(RESULTS_DIR.glob('*/comparison_*.png'))
metrics_files = sorted(RESULTS_DIR.glob('*/metrics.json'))
summary_csv = RESULTS_DIR / 'metrics_summary.csv'

if comparison_files:
    print('Comparison figures:')
    for path in comparison_files:
        print(' -', path.relative_to(ROOT))
    display(Image(filename=str(comparison_files[0])))
else:
    display(Markdown('Run evaluation script to generate results'))
    print('python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 5')

if metrics_files:
    rows = []
    for path in metrics_files:
        data = json.loads(path.read_text())
        model_name = path.parent.name
        rows.append({
            'model': model_name,
            'rmse': data.get('rmse'),
            'mae': data.get('mae'),
            'bias': data.get('bias'),
            'num_samples': data.get('num_samples'),
        })
    display(pd.DataFrame(rows).sort_values('model'))

if summary_csv.exists():
    print('Summary CSV:', summary_csv.relative_to(ROOT))
    display(pd.read_csv(summary_csv))


## 6. Interpretation and Next Directions
Typical behavior in this baseline setting:
- temporal models often improve large-scale consistency and reduce static bias
- fine-scale terrain-linked extremes remain challenging

Next technical steps: multi-variable ERA5 conditioning, multimodal data fusion, and uncertainty-aware training objectives.
